# NiftyAlpha: Exploratory Data Analysis & Feature Engineering

## 1. Problem Description & Quantitative Motivation
Most algorithmic trading research and machine learning publications in literature focus predominantly on developed markets (primarily the US S&P 500 and Nasdaq). While foundational, quantitative methodologies developed for US equities frequently fail when naively transferred to emerging markets due to distinct structural dynamics, foreign exchange volatility, and external macroeconomic dependency.

### The Indian Equity Market (NSE & BSE)
The Indian equity ecosystem, centered on the **National Stock Exchange (NSE)** and **Bombay Stock Exchange (BSE)**, offers deep institutional liquidity paired with dynamic macro sensitivities:
1. **Crude Oil Exposure**: India imports more than 80% of its domestic crude oil requirements. Shifts in global **Brent Crude** pricing propagate directly into currency pressure (USD/INR depreciation), imported inflation, input cost inflation for manufacturing, and central bank interest rate actions.
2. **Dual-Exchange Microstructure**: Many top Indian blue-chips trade concurrently on both the NSE and BSE. Volume divergences between the primary venue (NSE) and secondary venue (BSE) provide unique alternative data signals regarding institutional order routing.
3. **Union Budget Seasonality**: February marks the presentation of the Indian Union Budget, which introduces concentrated fiscal, tax, and regulatory volatility.

### Machine Learning Objective
The core objective of **NiftyAlpha** is to build a machine learning classification engine that predicts whether a top liquid Indian stock will achieve **positive growth over a 20-trading-day holding horizon** (approximately one calendar month):

$$\text{Target Return}_{t, t+20} = \frac{\text{Close}_{t+20} - \text{Close}_t}{\text{Close}_t}$$

$$\text{is\_positive\_growth} = \mathbb{I}(\text{Target Return}_{t, t+20} > 0)$$

In quantitative trade execution, false positives incur severe capital drawdown and bid-ask/tax friction. Therefore, our optimization criterion focuses strictly on **Precision** (minimizing false buy signals) rather than unweighted accuracy.

### Benchmark Definition
All predictive alpha and simulated portfolio trajectories are evaluated directly against a Buy & Hold benchmark on the **Nifty 50 Index (`^NSEI`)**, the definitive benchmark for large-cap Indian equities.


In [ ]:
# Import core numerical, financial, and data science libraries
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Visual style setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 150


## 2. Multi-Source Data Ingestion Strategy

To establish a holistic multi-asset quantitative regime, NiftyAlpha ingests three synchronized data universes:

1. **Primary Equities (NSE)**: Top liquid blue-chip equities from the Nifty 50 universe (e.g., Reliance Industries `RELIANCE.NS`, Tata Consultancy Services `TCS.NS`, HDFC Bank `HDFCBANK.NS`).
2. **Alternative Data (BSE Divergence)**: Equivalent listings from the Bombay Stock Exchange (`.BO`). Volume divergence between exchanges highlights localized institutional participation.
3. **Macroeconomic Conditioning (Global Commodities & Currencies)**:
   - **Brent Crude (`BZ=F`)**: Measures energy import cost pressure.
   - **Gold (`GC=F`)**: Gauges domestic wealth preservation and global risk-off hedging.
   - **USD / INR (`INR=X`)**: Reflects foreign institutional investment (FII) flows and currency risk.
   - **India VIX (`^INDIAVIX`)**: Direct gauge of annualized implied volatility and market regime stress.


In [ ]:
# Historical data download for representative assets via yfinance
tickers = ['RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', '^NSEI', 'BZ=F']
start_date = '2015-01-01'

print(f"Downloading historical daily data from {start_date} for: {tickers}\n")
raw_market_data = {}

for ticker in tickers:
    df = yf.download(ticker, start=start_date, progress=False, auto_adjust=False)
    
    # Flatten MultiIndex columns if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
        
    # Strip exchange-specific timezones to ensure timezone-naive calendar alignment
    if isinstance(df.index, pd.DatetimeIndex):
        df.index = df.index.tz_localize(None)
        
    raw_market_data[ticker] = df
    print(f"  [✓] {ticker:<14} | {len(df):>5} bars | {df.index.min().date()} -> {df.index.max().date()}")


## 3. Calendar Mismatches & Missing Value Imputation

### The Holiday Asymmetry Problem
When combining domestic Indian equities with international commodities and benchmark indices, calendar misalignment is inevitable:
- **Indian National Holidays**: NSE closes for festivals such as Diwali, Holi, and Indian Republic Day, while global commodities (Brent Crude in London/NYMEX) remain active.
- **Foreign Market Holidays**: Global commodities close for US Thanksgiving, Christmas, or UK Bank Holidays, while NSE remains open for normal trading.

### Preserving Lookahead-Free Integrity via Forward-Fill
Merging global series onto domestic trading dates introduces `NaN` values on days when international markets were closed.
- **Why NOT interpolate or backward-fill?** Imputing missing values using future prices (`bfill()` or linear interpolation) injects catastrophic **lookahead bias**, exposing the model to information that did not exist on trading day $t$.
- **Why forward-fill (`ffill()`)?** In reality, market participants at day $t$ can only reference the most recent publicly settled closing price from $t-1$. Forward-filling accurately reflects market reality without data leakage.


In [ ]:
# Alignment demonstration: Merging Nifty 50 and Brent Crude onto Reliance Industries
reliance_df = raw_market_data['RELIANCE.NS'][['Open', 'High', 'Low', 'Close', 'Volume']].copy()
reliance_df.rename(columns={'Close': 'Close_RELIANCE', 'Volume': 'Volume_RELIANCE'}, inplace=True)

# Context DataFrame: Nifty 50 and Brent Crude
macro_context = pd.DataFrame({
    'Close_NIFTY': raw_market_data['^NSEI']['Close'],
    'Close_BRENT': raw_market_data['BZ=F']['Close']
})

# Perform left-join on trading dates of the primary equity
merged_df = reliance_df.join(macro_context, how='left')

print("=================================================================")
print(" MISSING VALUE ANALYSIS: BEFORE FORWARD-FILL")
print("=================================================================")
print(merged_df[['Close_RELIANCE', 'Close_NIFTY', 'Close_BRENT']].isna().sum())

# Apply forward-fill to carry forward latest available prices without lookahead bias
merged_df['Close_NIFTY'] = merged_df['Close_NIFTY'].ffill().bfill()
merged_df['Close_BRENT'] = merged_df['Close_BRENT'].ffill().bfill()

print("\n=================================================================")
print(" MISSING VALUE ANALYSIS: AFTER FORWARD-FILL (ffill)")
print("=================================================================")
print(merged_df[['Close_RELIANCE', 'Close_NIFTY', 'Close_BRENT']].isna().sum())


In [ ]:
# Comparative Time-Series Visualization: Reliance Industries vs. Nifty 50 Index
norm_reliance = (merged_df['Close_RELIANCE'] / merged_df['Close_RELIANCE'].iloc[0]) * 100.0
norm_nifty = (merged_df['Close_NIFTY'] / merged_df['Close_NIFTY'].iloc[0]) * 100.0

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(14, 9), sharex=True, gridspec_kw={'height_ratios': [3, 1]}
)

# Plot 1: Rebased Performance Comparison (Base = 100)
ax1.plot(merged_df.index, norm_reliance, label='Reliance Industries (RELIANCE.NS)', color='#1f77b4', linewidth=2)
ax1.plot(merged_df.index, norm_nifty, label='Nifty 50 Benchmark (^NSEI)', color='#ff7f0e', linewidth=2, linestyle='--')
ax1.set_title('Reliance Industries vs. Nifty 50 Benchmark (Normalized Performance, Base = 100)', fontsize=14, fontweight='bold', pad=12)
ax1.set_ylabel('Normalized Valuation Index', fontsize=12)
ax1.legend(loc='upper left', frameon=True, fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.6)

# Plot 2: Relative Strength Ratio
relative_strength = merged_df['Close_RELIANCE'] / merged_df['Close_NIFTY']
ax2.plot(merged_df.index, relative_strength, color='#2ca02c', linewidth=1.5, label='Relative Ratio (Reliance / Nifty 50)')
ax2.axhline(relative_strength.mean(), color='gray', linestyle=':', label='Historical Mean Ratio')
ax2.set_title('Reliance Relative Strength to Benchmark', fontsize=12, fontweight='bold')
ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylabel('Relative Ratio', fontsize=12)
ax2.legend(loc='upper left', frameon=True, fontsize=10)
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


## 4. Quantitative Feature Engineering & Leakage Prevention

### Target Formulation
Our prediction objective is the forward 20-trading-day growth:
$$\text{target\_30d\_return} = \frac{\text{Close}_{t+20} - \text{Close}_t}{\text{Close}_t}$$
$$\text{is\_positive\_growth} = (\text{target\_30d\_return} > 0.0) \in \{0, 1\}$$

### Strict Temporal Boundaries
1. **Backward-Looking Features**:
   - Technical Indicators (RSI, Moving Average ratios, Volatility) must use historical data strictly up to day $t$ ($[t-W, t]$).
   - Cross-Market Sensitivities (rolling correlations between stock returns and Brent Crude returns) operate over past historical windows only.
2. **Forward-Looking Target**:
   - Computed via negative shift (`shift(-20)`).
3. **Partition Boundary Pruning**:
   - The initial $W$ burn-in days of rolling calculations and the final 20 trading days of the dataset (where forward return is unobservable) MUST be purged before model training to eliminate boundary lookahead leakage.


In [ ]:
# Calculate Target Variables and Predictive Feature Set

# 1. Target Variables (20 Trading Days Forward)
merged_df['target_30d_return'] = (merged_df['Close_RELIANCE'].shift(-20) - merged_df['Close_RELIANCE']) / merged_df['Close_RELIANCE']
merged_df['is_positive_growth'] = (merged_df['target_30d_return'] > 0.0).astype(int)

# 2. Technical Indicators (Backward Looking)
# 20-day Simple Moving Average & Ratio
merged_df['sma_20'] = merged_df['Close_RELIANCE'].rolling(window=20).mean()
merged_df['price_to_sma20'] = merged_df['Close_RELIANCE'] / merged_df['sma_20']

# 14-day Relative Strength Index (RSI via Wilder's Smoothing)
delta = merged_df['Close_RELIANCE'].diff()
gain = delta.clip(lower=0.0)
loss = -delta.clip(upper=0.0)
avg_gain = gain.ewm(alpha=1.0 / 14, min_periods=14, adjust=False).mean()
avg_loss = loss.ewm(alpha=1.0 / 14, min_periods=14, adjust=False).mean()
rs = avg_gain / (avg_loss + 1e-9)
merged_df['rsi_14'] = 100.0 - (100.0 / (1.0 + rs))

# 3. Macro Sensitivity Feature (30-day Rolling Correlation with Brent Crude Returns)
reliance_ret = merged_df['Close_RELIANCE'].pct_change()
brent_ret = merged_df['Close_BRENT'].pct_change()
merged_df['corr_brent_30d'] = reliance_ret.rolling(window=30).corr(brent_ret)

# Drop burn-in rows and forward unobservable bounds
clean_research_df = merged_df.dropna().copy()

print(f"Dataset summary after pruning boundary NaNs: {len(clean_research_df)} clean rows")
print("\nTail samples of engineered research features & target classification labels:")

display_columns = [
    'Close_RELIANCE', 'Close_BRENT', 'rsi_14', 'price_to_sma20',
    'corr_brent_30d', 'target_30d_return', 'is_positive_growth'
]
display_tail = clean_research_df[display_columns].tail(10)
print(display_tail.to_string())
